<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 7 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Resolve Out-of-Order Events, Preserve History, and Replay Recoverably</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">Handle duplicate and out-of-order order events, recover interrupted steps, and reconcile business transaction records.</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">Target Doris 4.1.3 · Order data · Dedicated lab database</span>
</div>

Complete Module 6 first. By the end, you will verify eleven current orders and eighteen logical history records after out-of-order and duplicate deliveries, and test partial column updates and deletion on independent copies. Run the cells in order.

[Course notes](course7_updates_deletes_and_replay.md) · [Course home](../README.md)


## Prerequisites and Business Sources

Complete Module 6 first. This lab reads accepted orders from orders_clean and processes simulated changes to orders 900001–900011 with source COURSE_SIMULATION.

Rebuild only orders_update_walkthrough, orders_replay_practice, orders_current, order_events, event_deliveries, orders_partial_update, orders_delete_demo, order_items, products, payments, refunds, and shipment_events.

Normal flow: created → paid → shipped → delivered; refund flow: created → paid → cancelled → refunded.
This lab reads simulated events from a file, delivering the delivered event before the paid event. Each event carries a complete updated state (after-image); duplicate deliveries of the same event carry identical content.

If wwi_products is missing, the business-ledger setup loads its 227 rows from the downloaded WWI Parquet package. An existing wwi_products table is reused without modification.


In [ ]:
from pathlib import Path
import sys
from uuid import uuid4

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows
from dw_course.ui import show_sql, show_response
from dw_course.wwi import manifest, parquet_ddl, parquet_paths

lab = connect_sandbox()




## Start with a Small Example: Normal Update, Duplicate, and Out-of-Order Events

This step rebuilds only orders_update_walkthrough, preserving Module 6's accepted table. First observe an order changing from CREATED to PAID, repeat the same event, then receive SHIPPED and a late PAID event. Query the current state after every step.

First inspect the printed table creation SQL and find these three settings:

| Setting | Purpose |
| --- | --- |
| UNIQUE KEY(order_id) | Identifies the same order |
| "enable_unique_key_merge_on_write"="true" | Uses the merge-on-write model |
| "function_column.sequence_col"="event_version" | Specifies the field used to compare business versions of the same order |

order_ddl(..., current=True) is a course helper that generates this DDL, not database syntax.
Adding an ordinary column named event_version alone does not enable version resolution; it must be configured as the Sequence column.
Each subsequent event provides a complete row and version. If version 3 arrives before version 2, version 3 is retained.
The same version with different content is not a safe redelivery in this lab; separate conflict rules are required.


In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_update_walkthrough")
ddl = order_ddl("orders_update_walkthrough", current=True)
show_sql("Inspect the primary key and Sequence configuration first", ddl)
lab.execute(ddl)
initial = dict(fixture("orders.json")[0])
lab.insert("orders_update_walkthrough", ORDER_COLUMNS, order_rows([initial]))
lab.sql("SELECT order_id,status,event_version FROM orders_update_walkthrough", title="Initial order")


### Normal Update

Version 2 represents completed payment. The new event carries the complete order state, with the order ID unchanged.


In [ ]:
paid = dict(initial, status="PAID", event_version=2, event_id="WALK_PAID", paid_amount="100.00")
lab.insert("orders_update_walkthrough", ORDER_COLUMNS, order_rows([paid]))
lab.sql("SELECT order_id,status,event_version,paid_amount FROM orders_update_walkthrough", title="Current state after payment")
expect(lab.query("SELECT status,event_version FROM orders_update_walkthrough"), [("PAID",2)])


### Duplicate Delivery

Write the same payment event again. Predict whether the order will gain another row or remain one row, then inspect the result.


In [ ]:
lab.insert("orders_update_walkthrough", ORDER_COLUMNS, order_rows([paid]))
lab.sql("SELECT order_id,status,event_version,paid_amount FROM orders_update_walkthrough", title="The same event arrives again")
expect(lab.query("SELECT COUNT(*),MAX(event_version) FROM orders_update_walkthrough"), [(1,2)])


### Out-of-Order Delivery

Process version 3's shipment first, then redeliver version 2's older payment. The Sequence column selects the current state by business version.


In [ ]:
shipped = dict(paid, status="SHIPPED", event_version=3, event_id="WALK_SHIPPED")
lab.insert("orders_update_walkthrough", ORDER_COLUMNS, order_rows([shipped]))
lab.insert("orders_update_walkthrough", ORDER_COLUMNS, order_rows([paid]))
lab.sql("SELECT order_id,status,event_version,paid_amount FROM orders_update_walkthrough", title="The late payment event did not overwrite the shipped state")
expect(lab.query("SELECT status,event_version FROM orders_update_walkthrough"), [("SHIPPED",3)])


## 1. Create Separate Current-State and History Tables

The current table uses UNIQUE KEY(order_id) and retains the newer state by event_version; the history table uses UNIQUE KEY(event_id) and retains distinct events; the delivery table records every attempt.

These tables are written in separate steps. Next, observe recovery when one table has been updated but another has not.


In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_current")
ddl = order_ddl("orders_current", current=True)
show_sql("Table creation SQL", ddl)
lab.execute(ddl)
lab.execute("DROP TABLE IF EXISTS order_events")
ddl = order_ddl("order_events", history=True)
show_sql("Table creation SQL", ddl)
lab.execute(ddl)
lab.execute(f"INSERT INTO orders_current ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM orders_clean")
lab.execute(f"INSERT INTO order_events ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM orders_clean")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_current"), [(10,"1400.00")])
lab.execute("DROP TABLE IF EXISTS event_deliveries")
lab.execute('CREATE TABLE event_deliveries (attempt_id BIGINT, delivery_id BIGINT, event_id VARCHAR(32), payload STRING) DUPLICATE KEY(attempt_id, delivery_id) DISTRIBUTED BY HASH(attempt_id) BUCKETS 1 PROPERTIES("replication_num"="1")')


## 2. Simulate an Interruption, Then Recover

Record the first delivery and write history, then simulate an interruption before writing the current table. Deliver the entire event batch again, preserving each delivery and deduplicating history by stable event ID. This experiment interrupts only the course steps, not the database process.


In [ ]:
import json
deliveries = fixture("deliveries.json")
first = deliveries[0]
lab.insert("event_deliveries", ["attempt_id","delivery_id","event_id","payload"],
           [(0, first["delivery_id"], first["event_id"], json.dumps(first))])
lab.insert("order_events", ORDER_COLUMNS, order_rows([first]))
expect(lab.query("SELECT status FROM orders_current WHERE order_id=900001"), [("CREATED",)])

def replay(attempt):
    for record in deliveries:
        lab.insert("event_deliveries", ["attempt_id","delivery_id","event_id","payload"],
                   [(attempt, record["delivery_id"], record["event_id"], json.dumps(record))])
        lab.insert("order_events", ORDER_COLUMNS, order_rows([record]))
        lab.insert("orders_current", ORDER_COLUMNS, order_rows([record]))

replay(1)


## 3. Reconcile Row by Row, Then Replay the Entire Batch

900001 remains DELIVERED at version 4; 900003 remains REFUNDED at version 4 with cumulative payments still at 150.00.
10 initial snapshots plus 8 distinct events produce 18 logical history records and 11 current orders.
Each batch has 9 deliveries including one duplicate; 1 delivery before interruption plus two full-batch replays gives 19 raw deliveries.


In [ ]:
projection = ",".join("DATE_FORMAT(event_time, '%Y-%m-%d %H:%i:%s')" if col == "event_time" else col for col in ORDER_COLUMNS)
def verify_state():
    expect(lab.query(f"SELECT {projection} FROM orders_current ORDER BY order_id"), order_rows(fixture("expected_current.json")))
    expect(lab.query("SELECT COUNT(*), SUM(order_amount), SUM(paid_amount), SUM(refund_amount) FROM orders_current"),
           [(11,"1510.00","250.00","150.00")])
    expect(lab.query("SELECT COUNT(*) FROM order_events"), [(18,)])
    expected_history = {r["event_id"]:r for r in fixture("orders.json") + deliveries}
    expect(lab.query(f"SELECT {projection} FROM order_events ORDER BY event_id"),
           order_rows([expected_history[key] for key in sorted(expected_history)]))
verify_state()
replay(2)
verify_state()
expect(lab.query("SELECT COUNT(*) FROM event_deliveries"), [(19,)])
lab.sql("SELECT 'Current orders' AS object,COUNT(*) AS records FROM orders_current UNION ALL SELECT 'Business history',COUNT(*) FROM order_events UNION ALL SELECT 'Raw deliveries',COUNT(*) FROM event_deliveries", title="Counts for each of the three record types")


### Independently Verify Current State with Business Transaction Records

Reconcile against the separately listed items, payments, refunds, and shipment events in business_events.json: item totals should equal order amounts, payments should total 250.00, and refunds 150.00. Payments and refunds are deduplicated by their own business IDs.

Refunds must link to original payments, and shipment events to orders. The product dimension comes from Module 5's wwi_products (227 rows). After writing the same business transaction records again, distinct business record counts and amounts should remain unchanged.

If wwi_products is missing, the business-ledger setup loads its 227 rows from the downloaded WWI Parquet package. An existing wwi_products table is reused without modification.


In [ ]:
business = fixture("business_events.json")
definitions = {
    "order_items": 'order_line_id BIGINT NOT NULL, order_id BIGINT, product_id BIGINT, quantity INT, unit_price DECIMAL(12,2)',
    "products": 'product_id BIGINT NOT NULL, product_name STRING',
    "payments": 'payment_id VARCHAR(32) NOT NULL, order_id BIGINT, event_id VARCHAR(32), amount DECIMAL(12,2), event_time DATETIME',
    "refunds": 'refund_id VARCHAR(32) NOT NULL, payment_id VARCHAR(32), order_id BIGINT, event_id VARCHAR(32), amount DECIMAL(12,2), event_time DATETIME',
    "shipment_events": 'shipment_event_id VARCHAR(32) NOT NULL, shipment_id VARCHAR(32), order_id BIGINT, event_id VARCHAR(32), status VARCHAR(20), event_time DATETIME',
}
for table, fields in definitions.items():
    key = fields.split()[0]
    lab.execute("DROP TABLE IF EXISTS " + table)
    ddl = f'CREATE TABLE {table} ({fields}) UNIQUE KEY({key}) DISTRIBUTED BY HASH({key}) BUCKETS 1 PROPERTIES("replication_num"="1", "enable_unique_key_merge_on_write"="true")'
    show_sql("Business transaction tables", ddl)
    lab.execute(ddl)
if not lab.query("SHOW TABLES LIKE 'wwi_products'"):
    paths = parquet_paths()
    metadata = manifest()["tables"]["products"]
    lab.execute(parquet_ddl("products", "wwi_products"))
    response = lab.stream_load(
        "wwi_products", paths["products"], "module7_wwi_" + uuid4().hex, format="parquet"
    )
    show_response(response)
    expect(response["Status"], "Success")
    expect(response["NumberLoadedRows"], metadata["rows"])
    expect(response["NumberFilteredRows"], 0)

lab.execute("INSERT INTO products SELECT StockItemID, StockItemName FROM wwi_products")
expect(lab.query("SELECT COUNT(*) FROM products"), [(227,)])
for attempt in range(2):
    for table, records in [("order_items", business["order_lines"]), ("payments", business["payments"]),
                           ("refunds", business["refunds"]), ("shipment_events", business["shipments"])]:
        columns = list(records[0])
        lab.insert(table, columns, [tuple(r[c] for c in columns) for r in records])
expect(lab.query("SELECT COUNT(*), SUM(amount) FROM payments"), [(2, "250.00")])
expect(lab.query("SELECT COUNT(*), SUM(amount) FROM refunds"), [(1, "150.00")])
expect(lab.query("SELECT COUNT(*) FROM shipment_events"), [(2,)])
expect(lab.query("SELECT COUNT(*) FROM order_items i LEFT JOIN products p ON i.product_id=p.product_id WHERE p.product_id IS NULL"), [(0,)])
expect(lab.query("SELECT COUNT(*) FROM orders_current o LEFT JOIN customers c ON o.customer_id=c.customer_id WHERE c.customer_id IS NULL"), [(0,)])
expect(lab.query("""
SELECT COUNT(*) FROM refunds r LEFT JOIN payments p
ON r.payment_id=p.payment_id AND r.order_id=p.order_id
WHERE p.payment_id IS NULL OR r.amount>p.amount
"""), [(0,)])
for table in ("payments", "refunds", "shipment_events"):
    expect(lab.query(f"SELECT COUNT(*) FROM {table} b LEFT JOIN order_events h ON b.event_id=h.event_id AND b.order_id=h.order_id WHERE h.event_id IS NULL"), [(0,)])
expect(lab.query("""
SELECT COUNT(*) FROM orders_current o
LEFT JOIN (SELECT order_id, SUM(quantity*unit_price) amount FROM order_items GROUP BY order_id) i ON o.order_id=i.order_id
LEFT JOIN (SELECT order_id, SUM(amount) paid FROM payments GROUP BY order_id) p ON o.order_id=p.order_id
LEFT JOIN (SELECT order_id, SUM(amount) refunded FROM refunds GROUP BY order_id) r ON o.order_id=r.order_id
WHERE i.order_id IS NULL OR o.order_amount<>i.amount
   OR o.paid_amount<>COALESCE(p.paid,0) OR o.refund_amount<>COALESCE(r.refunded,0)
"""), [(0,)])
lab.sql("SELECT order_id, status, event_version, paid_amount, refund_amount, data_source FROM orders_current ORDER BY order_id", title="Current order state and source")


## 4. Partial Column Updates

Temporarily enable partial column updates on the independent orders_partial_update table, submitting the order ID, new status, and increasing version. Verify that the status becomes CANCELLED while the amount remains 100.00 and the region EAST. Restore session settings afterward.


In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_partial_update")
ddl = order_ddl("orders_partial_update", current=True)
show_sql("Table creation SQL", ddl)
lab.execute(ddl)
lab.insert("orders_partial_update", ORDER_COLUMNS, order_rows(fixture("orders.json")[:1]))
previous_partial = lab.query("SELECT @@enable_unique_key_partial_update")[0][0]
try:
    lab.execute("SET enable_unique_key_partial_update = true")
    lab.execute("INSERT INTO orders_partial_update (order_id, status, event_version) VALUES (900001, 'CANCELLED', 2)")
finally:
    lab.execute("SET enable_unique_key_partial_update = %s", (previous_partial,))
expect(lab.query("SELECT status, event_version, order_amount, region FROM orders_partial_update"),
       [("CANCELLED",2,"100.00","EAST")])


## 5. Independent Deletion Experiment

Do not delete refunded orders in the main workflow. In a copy, use is_deleted for business soft deletion, then SQL DELETE for another row; query invisibility does not mean immediate disk reclamation.


In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_delete_demo")
lab.execute('CREATE TABLE orders_delete_demo (order_id BIGINT, is_deleted BOOLEAN, status VARCHAR(20)) UNIQUE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1 PROPERTIES("replication_num"="1", "enable_unique_key_merge_on_write"="true")')
lab.insert("orders_delete_demo", ["order_id","is_deleted","status"], [(900001,False,"CREATED"),(900002,False,"CREATED")])
lab.execute("UPDATE orders_delete_demo SET is_deleted=true WHERE order_id=900001")
expect(lab.query("SELECT COUNT(*) FROM orders_delete_demo"), [(2,)])
expect(lab.query("SELECT order_id FROM orders_delete_demo WHERE is_deleted=false"), [(900002,)])
lab.execute("DELETE FROM orders_delete_demo WHERE order_id=900002")
expect(lab.query("SELECT order_id FROM orders_delete_demo"), [(900001,)])
verify_state()


## Completion and Your Turn

Verify eleven current orders, eighteen logical history records, and nineteen raw deliveries, and check item details and payment/refund reconciliation results.

Display the history of order 900003 by event_time and explain why paid_amount remains 150.00 after the refund while net receipts are zero.


## Independent Exercise

Copy the current state of order 900001 to the independent orders_replay_practice table, then construct a complete old event with version 2 and status PAID and deliver it twice in a row. Predict the status, version, and row count before running and checking it. Expect one order, still DELIVERED at version 4.

This exercise rebuilds only orders_replay_practice; the main history and delivery tables remain unchanged.

Write and run your code in the next cell, then expand the reference solution after completing the exercise. A blank exercise is not automatically marked as complete.


In [ ]:
# Write your SQL or load request here.


<details>
<summary>Reference solution (expand after completion)</summary>

```python
lab.execute("DROP TABLE IF EXISTS orders_replay_practice")
ddl = order_ddl("orders_replay_practice", current=True)
show_sql("Independent replay exercise table", ddl)
lab.execute(ddl)
lab.execute("INSERT INTO orders_replay_practice SELECT * FROM orders_current WHERE order_id=900001")
late_event = """INSERT INTO orders_replay_practice
(order_id,customer_id,order_amount,status,event_version,event_id,event_time,paid_amount,refund_amount,region,data_source)
SELECT order_id,customer_id,order_amount,'PAID',2,'PRACTICE_OLD','2026-01-01 10:00:00',
paid_amount,refund_amount,region,data_source FROM orders_current WHERE order_id=900001"""
lab.execute(late_event)
lab.execute(late_event)
lab.sql("SELECT order_id,status,event_version,paid_amount FROM orders_replay_practice", title="Result after redelivering the late old event twice")
expect(lab.query("SELECT order_id,status,event_version,paid_amount FROM orders_replay_practice"), [(900001,"DELIVERED",4,"100.00")])
```

</details>
